In [1]:
# 3d bar one end fixed modal analysis

from dolfinx.fem import Function, functionspace, form, dirichletbc, locate_dofs_geometrical
from dolfinx.fem.petsc import assemble_matrix
from slepc4py import SLEPc
from ufl import dx, grad, inner, TrialFunction, TestFunction, sym, tr, Identity
from dolfinx.io import XDMFFile
from dolfinx.mesh import create_box, CellType
from mpi4py import MPI
import numpy as np

# -----------------------
# Geometry (blade-like 1:6:thin)
# -----------------------
L = 6.0
W = 1.0
T = 0.1   # thickness

nx, ny, nz = 60, 10, 3   # 3 elements through thickness

mesh = create_box(
    MPI.COMM_WORLD,
    [[0.0, 0.0, 0.0], [L, W, T]],
    [nx, ny, nz],
    cell_type=CellType.hexahedron
)

# -----------------------
# Material (steel)
# -----------------------
E = 210e9
nu = 0.3
rho = 7800

mu = E / (2 * (1 + nu))
lmbda = E * nu / ((1 + nu) * (1 - 2 * nu))

# -----------------------
# Function space (VECTOR 3D)
# -----------------------
deg = 1
V = functionspace(mesh, ("Lagrange", deg, (3,)))

u = TrialFunction(V)
v = TestFunction(V)

# -----------------------
# Elasticity definitions
# -----------------------
def eps(u):
    return sym(grad(u))

def sigma(u):
    return lmbda * tr(eps(u)) * Identity(3) + 2 * mu * eps(u)

# -----------------------
# Forms
# -----------------------
k_form = form(inner(sigma(u), eps(v)) * dx)
m_form = form(rho * inner(u, v) * dx)

# -----------------------
# Boundary condition (cantilever root)
# -----------------------
def left(x):
    return np.isclose(x[0], 0.0)

dofs = locate_dofs_geometrical(V, left)
bc = dirichletbc(np.array((0.0, 0.0, 0.0), dtype=np.complex128), dofs, V)

# -----------------------
# Assemble matrices
# -----------------------
K = assemble_matrix(k_form, bcs=[bc])
M = assemble_matrix(m_form, bcs=[bc])

K.assemble()
M.assemble()

# -----------------------
# Setup the eigensolver
# -----------------------
solver = SLEPc.EPS().create()
solver.setDimensions(15)
solver.setProblemType(SLEPc.EPS.ProblemType.GHEP)

st = SLEPc.ST().create()
st.setType(SLEPc.ST.Type.SINVERT)
st.setShift(1e-3)
st.setFromOptions()

solver.setST(st)
solver.setOperators(K, M)

# -----------------------
# solving the problem
# -----------------------
solver.solve()

xr, xi = K.createVecs()
tol, maxit = solver.getTolerances()
nconv = solver.getConverged()

print("Number of iterations of the method: %i" % solver.getIterationNumber())
print("Solution method: %s" % solver.getType())
print("")

print("Stopping condition: tol=%.4g, maxit=%d" % (tol, maxit))

eig_vector = []
eig_freq = []

countfn = 0
if nconv > 0:
    for i in range(nconv):
        k = solver.getEigenpair(i, xr, xi)

        if k.real > 0:
            fn = np.sqrt(k.real) / (2 * np.pi)
        else:
            fn = 0.0
        eig_freq.append(fn)

        print("%12f Hz" % fn)
        countfn = countfn + 1

        vect = xr.getArray().copy()
        eig_vector.append(vect)

print("")
print("countfn = " + str(countfn))

# -----------------------
# Write modes
# -----------------------
countXDMF = 0
for i in range(len(eig_vector)):
    with XDMFFile(mesh.comm, "3D-Mode_" + str(i+1) + "_" + str(np.round(eig_freq[i], 2)) + "_Hz.xdmf", "w") as xdmf:
        u = Function(V)
        u.x.array[:] = eig_vector[i]
        u.name = "u"
        countXDMF = countXDMF + 1

        xdmf.write_mesh(mesh)
        xdmf.write_function(u)

print("")
print("countXDMF = " + str(countXDMF))

Number of iterations of the method: 2
Solution method: krylovschur

Stopping condition: tol=1e-08, maxit=536
    0.159155 Hz
    0.159155 Hz
    0.159155 Hz
    0.159155 Hz
    0.159155 Hz
    0.159155 Hz
    0.159155 Hz
    0.159155 Hz
    0.159155 Hz
    0.159155 Hz
    0.159155 Hz
    0.159155 Hz
    2.799066 Hz
   17.518626 Hz
   22.946933 Hz
   27.441738 Hz
   49.025095 Hz
   83.873265 Hz
   96.035763 Hz
  128.928978 Hz
  144.765161 Hz
  158.674150 Hz

countfn = 22

countXDMF = 22
